# DNABERT2 Frozen Benchmark: ai x bio

This notebook runs only the DNABERT2 frozen embedding benchmark under the `benchmarks_sg` workflow. It prepares the Drive dataset named `ai x bio`, writes standardized `sequence,label,id` train/validation/test CSVs, and runs the frozen DNABERT2 benchmark with the same validation-only threshold policy used by CNN-v2.

Scientific rules:
- use train/validation/test CSVs only
- select threshold on validation MCC only
- report test metrics once using that validation-selected threshold
- do not tune on test
- do not fake skipped or failed model results


## 1. Setup repository

Run this first. It clones or refreshes the branch that contains the benchmark harness.


In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-all-model-baselines"
REPO_DIR = Path("/content/SeqTrainer")

if REPO_DIR.exists():
    %cd /content/SeqTrainer
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd /content/SeqTrainer

print("Repository:", REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!git rev-parse --short HEAD


## 2. Install SeqTrainer with torch dependencies

This makes `seqtrainer` importable in Colab and installs the Hugging Face/PyTorch path needed by DNABERT2.


In [ ]:
%cd /content/SeqTrainer
!python -m pip install --upgrade pip setuptools wheel -q
!python -m pip install -e ".[torch]" -q

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import seqtrainer
print("SeqTrainer:", seqtrainer.__file__)


## 3. Mount Drive

The prep script searches your mounted Drive for a file whose name looks like `ai x bio`. If you already know the exact file path, set `SOURCE_FILE` in the next cell.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Drive already mounted")
except Exception as exc:
    print("Drive mount skipped or failed:", exc)


## 4. Prepare ai x bio CSV splits

This writes the standardized files used by the DNABERT2 config:

- `data/benchmarks/ai_x_bio/train.csv`
- `data/benchmarks/ai_x_bio/validation.csv`
- `data/benchmarks/ai_x_bio/test.csv`

The source may be CSV, TSV, FASTA, or JSONL. If the source has a split column, it is preserved. Otherwise a seeded stratified split with seed `42` is created.


In [ ]:
%cd /content/SeqTrainer

DRIVE_ROOT = Path("/content/drive/MyDrive")
SOURCE_FILE = None  # Example: Path("/content/drive/MyDrive/AIxBio/ai x bio.csv")

cmd = [
    sys.executable,
    "notebooks/benchmarks_sg/prepare_ai_x_bio_splits.py",
    "--drive-root",
    str(DRIVE_ROOT),
    "--output-dir",
    "data/benchmarks/ai_x_bio",
    "--seed",
    "42",
]
if SOURCE_FILE is not None:
    cmd.extend(["--source-file", str(SOURCE_FILE)])

import subprocess
subprocess.run(cmd, check=True)


## 5. Inspect prepared splits

Before running a model, confirm the row counts, class counts, and sequence lengths.


In [ ]:
import json
import pandas as pd

DATA_DIR = REPO_DIR / "data" / "benchmarks" / "ai_x_bio"
for split in ("train", "validation", "test"):
    frame = pd.read_csv(DATA_DIR / f"{split}.csv")
    lengths = frame["sequence"].astype(str).str.len()
    print(f"\n{split}")
    print("rows:", len(frame))
    print("columns:", list(frame.columns))
    print("label counts:")
    print(frame["label"].value_counts().sort_index())
    print("sequence length min/median/max:", int(lengths.min()), float(lengths.median()), int(lengths.max()))

metadata = json.loads((DATA_DIR / "dataset_prep_metadata.json").read_text())
metadata


## 6. Verify DNABERT2 runner patch

This checks that Colab is using the package code with the pad-token and meta-device fallback helpers.


In [ ]:
import inspect
import importlib
import seqtrainer.torch.dnabert2_benchmark as dnabert2_benchmark

importlib.reload(dnabert2_benchmark)
source_text = inspect.getsource(dnabert2_benchmark)
print("DNABERT2 runner file:", dnabert2_benchmark.__file__)
print("Has pad-token setter:", "_set_pad_token_id" in source_text)
print("Has meta-device fallback:", "_load_dnabert2_from_state_dict" in source_text)
print("Has meta runtime check:", "_is_meta_device_error" in source_text)


## 7. Run frozen DNABERT2 benchmark

This downloads `zhihan1996/DNABERT-2-117M` if needed, extracts frozen embeddings, trains only the small classifier head, and saves the shared benchmark artifacts.

If this fails due to Hugging Face rate limits, add an `HF_TOKEN` in Colab secrets and restart runtime.


In [ ]:
from seqtrainer.benchmarks.runner import run_benchmark

CONFIG = REPO_DIR / "config-examples" / "benchmarks" / "dnabert2_ai_x_bio_frozen.toml"
result = run_benchmark(CONFIG, base_dir=REPO_DIR, allow_skip=False)
print("status:", result.status)
print("output_dir:", result.output_dir)


## 8. Display metrics

Main metric to watch: test MCC. Secondary metric: test AUPRC.


In [ ]:
metrics_path = Path(result.output_dir) / "metrics.csv"
metrics = pd.read_csv(metrics_path)
metrics

cols = ["split", "threshold", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "mcc", "auroc", "auprc", "loss"]
metrics[[col for col in cols if col in metrics.columns]]


## 9. Inspect artifacts

A completed run should contain metrics, predictions, manifest, history, checkpoints, and cached embeddings.


In [ ]:
!find outputs/benchmarks/dnabert2_frozen_ai_x_bio -maxdepth 3 -type f | sort

manifest = json.loads((Path(result.output_dir) / "manifest.json").read_text())
manifest["model"]["metadata"]


## 10. Download outputs

Download this zip from the Colab file browser if you need to share the run artifacts.


In [ ]:
%cd /content/SeqTrainer
!zip -qr /content/dnabert2_ai_x_bio_outputs.zip outputs/benchmarks/dnabert2_frozen_ai_x_bio data/benchmarks/ai_x_bio
print("Created /content/dnabert2_ai_x_bio_outputs.zip")
